In [ ]:
import pandas as pd
import yfinance as yf
from edgar import *
from IPython.display import display
import os
from dotenv import load_dotenv

load_dotenv()
set_identity(os.getenv("EMAIL"))
print("All imported")

In [230]:
def get_companys_datas(acq, tgt, verbose=False):

    acq_dict = yf.Ticker(acq).info

    tgt_dict = yf.Ticker(tgt).info

    if verbose == False:
        pass
    else:
        print(f"{'values':<15} | {acq_dict['symbol']:<25} | {tgt_dict['symbol']:<25}")
        print("-" * 60)
        print(
            f"{'Market cap':<15} | {round(acq_dict['marketCap']):<25,} | {round(tgt_dict['marketCap']):<25,}"
        )
        print(
            f"{'Price':<15} | {acq_dict['previousClose']:<25} | {tgt_dict['previousClose']:<25}"
        )
        print(
            f"{'Shares':<15} | {acq_dict['sharesOutstanding']:<25} | {tgt_dict['sharesOutstanding']:<25}\n\n"
        )

    return acq_dict, tgt_dict


get_companys_datas("AAPL", "NCLH")

({'address1': 'One Apple Park Way',
  'city': 'Cupertino',
  'state': 'CA',
  'zip': '95014',
  'country': 'United States',
  'phone': '(408) 996-1010',
  'website': 'https://www.apple.com',
  'industry': 'Consumer Electronics',
  'industryKey': 'consumer-electronics',
  'industryDisp': 'Consumer Electronics',
  'sector': 'Technology',
  'sectorKey': 'technology',
  'sectorDisp': 'Technology',
  'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover an

In [192]:
def contract_offer(
    acq_dict,
    tgt_dict,
    offer_premium=0.60,
    stock_pct=0.50,
    tax_rate=0.40,
    years=5,
    interest_rate=0.05,
    financing_fees_pct=0.035,
    transaccion_fees_pct=0.02,
    synergies_pct=0,
    amortization_years=10,
    verbose=False,
):

    # % of deal pay by cahs and with stocks
    cash_pct = 1 - stock_pct

    # Amount of money the acusition will cost
    share_price = tgt_dict["previousClose"] * (1 + offer_premium)
    offer_value = tgt_dict["sharesOutstanding"] * share_price
    acq_issued_shares = offer_value / acq_dict["previousClose"] * stock_pct

    # All expenses and writre offs
    # Transaccion fees:
    transaccion_fees = offer_value * transaccion_fees_pct
    # Financing fees:
    acq_borrowing = offer_value * cash_pct
    financing_fees = acq_borrowing * financing_fees_pct
    financing_fees_amort = financing_fees / years
    # Posible write of
    synergies = tgt_dict["totalRevenue"] * synergies_pct
    asset_write_off = (offer_value - tgt_dict["bookValue"]) * 0.15
    incremental_DA_expense = asset_write_off / amortization_years

    # Expected earnings
    acq_implide_net_inc = acq_dict["sharesOutstanding"] * acq_dict["epsCurrentYear"]
    tgt_implide_net_inc = tgt_dict["sharesOutstanding"] * tgt_dict["epsCurrentYear"]
    acq_implide_pretax_inc = acq_implide_net_inc / (1 - tax_rate)
    tgt_implide_pretax_inc = tgt_implide_net_inc / (1 - tax_rate)

    # Getting all together and in negatice (accounting reasons)
    profroma_pretax_unadj = acq_implide_pretax_inc + tgt_implide_pretax_inc
    interest_expense_deal = acq_borrowing * interest_rate * -1
    incremental_DA_expense *= -1
    transaccion_fees *= -1
    financing_fees_amort *= -1
    synergies *= 1

    # after merge
    profroma_pretax_adj = (
        profroma_pretax_unadj
        + interest_expense_deal
        + incremental_DA_expense
        + transaccion_fees
        + financing_fees_amort
        + synergies
    )
    proforma_net_income = profroma_pretax_adj * (1 - tax_rate)
    proforma_shares_outstanding = acq_dict["sharesOutstanding"] + acq_issued_shares
    proforma_eps = proforma_net_income / proforma_shares_outstanding
    accretion_dilution_per_share = proforma_eps - acq_dict["epsCurrentYear"]
    accretion_dilution_pct = (proforma_eps / acq_dict["epsCurrentYear"]) - 1

    if verbose == False:
        pass
    else:
        print("Parameters:")
        print(f"Offer premium:            | {offer_premium * 100}%")
        print(f"% of cash:                | {cash_pct * 100}%")
        print(f"% of stock:               | {stock_pct * 100}%")
        print(f"tax rate:                 | {tax_rate * 100}%")
        print(f"interest rate:            | {interest_rate * 100}%")
        print(f"% financing fees:         | {transaccion_fees_pct * 100}%")
        print(f"% transaccion fees:       | {transaccion_fees_pct * 100}%")
        print(f"% synergies:              | {synergies_pct * 100}%")
        print(f"{'-' * 60}\n")

        print("Deal:")
        print(f"Share price:              | {round(share_price):,}")
        print(f"Offer Value:              | {round(offer_value):,}")
        print(f"Money borrowed:           | {round(acq_borrowing):,}")
        print(f"Financing fees:           | {round(financing_fees):,}")
        print(f"Shares issued:            | {round(acq_issued_shares):,}\n")
        print(f"{'-' * 60}\n")

        print("Income:")
        print(f"Accuary net income:       | {round(acq_implide_net_inc):,}")
        print(f"Target net income:        | {round(tgt_implide_net_inc):,}")
        print(f"Accuary pre tax income:   | {round(acq_implide_pretax_inc):,}")
        print(f"Target pre tax income:    | {round(tgt_implide_pretax_inc):,}")
        print(f"{'-' * 60}\n")

        print("Totals")
        print(f"Proforma pretax unadj:    | {round(profroma_pretax_unadj):,}")
        print(f"Interest expenses:        | ({round(interest_expense_deal * -1):,})")
        print(f"Amort of finance fees:    | ({round(financing_fees_amort * -1):,})")
        print(f"Transaccion fees:         | ({round(transaccion_fees * -1):,})")
        print(f"D/A write off:            | ({round(incremental_DA_expense * -1):,})")
        print(f"% synergies:              | {round(synergies):,}")
        print(f"{'-' * 60}\n")

        print("After merge:")
        print(f"Proforma pretax adj:      | {round(profroma_pretax_adj):,}")
        print(f"Proforma Net:             | {round(proforma_net_income):,}")
        print(f"Proforma shares:          | {round(proforma_shares_outstanding):,}")
        print(f"Proforma eps:             | {round(proforma_eps, ndigits=2)}")
        print(f"{'-' * 60}\n")

        print("Results")
        print(f"Accretion/Dilution:       | $ {accretion_dilution_per_share:.2f}")
        print(f"%Accretion/Dilution:      | % {accretion_dilution_pct:.2f}")

    return accretion_dilution_pct

In [189]:
acq, tgt = get_companys_datas("AAPL", "SPOT")

contract_offer(acq, tgt, offer_premium=0.30, verbose=False)

values          | AAPL                      | SPOT                     
------------------------------------------------------------
Market cap      | 3,644,938,780,672         | 97,704,951,808           
Price           | 248.96                    | 482.52                   
Shares          | 14681140000               | 205832527                




-0.03492217175138612

In [272]:
def highlight_irr(val):
    if val >= 0:
        return "background-color: #d4edda !important; color: black !important"
    elif val >= -0.05:
        return "background-color: #fff3cd !important; color: black !important"
    else:
        return "background-color: #f8d7da !important; color: black !important"


def sensitivity_accretion_dilution(acq, tgt, verbose=False, steps=1):

    rows = []
    for i in range(0, 11, steps):
        row = []
        for j in range(0, 11, steps):
            accretion_dilution_pct = round(
                contract_offer(
                    acq, tgt, offer_premium=i / 10, stock_pct=j / 10, verbose=False
                ),
                ndigits=2,
            )

            row.append(accretion_dilution_pct)

        rows.append(row)

    df = df = pd.DataFrame(
        rows,
        columns=[f"Stock {x * 10}%" for x in range(0, 11, steps)],
        index=[f"Offer premium {x * 10}%" for x in range(0, 11, steps)],
    )
    styled = df.style.format("{:.2f}%").map(highlight_irr)
    display(styled)

    return df

In [273]:
def highlight_irr(val):
    if val >= 0:
        return "background-color: #d4edda !important; color: black !important"
    elif val >= -0.05:
        return "background-color: #fff3cd !important; color: black !important"
    else:
        return "background-color: #f8d7da !important; color: black !important"

In [274]:
acq, tgt = get_companys_datas("SONO", "NCLH")
sensitivity_accretion_dilution(acq, tgt, verbose=False, steps=1)

,Stock 0%,Stock 10%,Stock 20%,Stock 30%,Stock 40%,Stock 50%,Stock 60%,Stock 70%,Stock 80%,Stock 90%,Stock 100%
Offer premium 0%,4.31%,2.57%,1.74%,1.26%,0.94%,0.72%,0.55%,0.42%,0.32%,0.24%,0.17%
Offer premium 10%,3.94%,2.23%,1.46%,1.02%,0.74%,0.54%,0.39%,0.28%,0.19%,0.12%,0.06%
Offer premium 20%,3.57%,1.92%,1.21%,0.81%,0.56%,0.38%,0.25%,0.15%,0.08%,0.01%,-0.04%
Offer premium 30%,3.21%,1.62%,0.97%,0.62%,0.39%,0.24%,0.13%,0.04%,-0.02%,-0.08%,-0.12%
Offer premium 40%,2.84%,1.35%,0.76%,0.45%,0.25%,0.12%,0.02%,-0.05%,-0.11%,-0.16%,-0.19%
Offer premium 50%,2.47%,1.09%,0.57%,0.29%,0.12%,0.01%,-0.07%,-0.14%,-0.18%,-0.22%,-0.26%
Offer premium 60%,2.10%,0.84%,0.39%,0.15%,0.01%,-0.09%,-0.16%,-0.21%,-0.25%,-0.29%,-0.31%
Offer premium 70%,1.73%,0.61%,0.22%,0.02%,-0.10%,-0.18%,-0.24%,-0.28%,-0.31%,-0.34%,-0.36%
Offer premium 80%,1.37%,0.39%,0.07%,-0.10%,-0.19%,-0.26%,-0.31%,-0.34%,-0.37%,-0.39%,-0.41%
Offer premium 90%,1.00%,0.19%,-0.07%,-0.20%,-0.28%,-0.33%,-0.37%,-0.40%,-0.42%,-0.43%,-0.45%


,Stock 0%,Stock 10%,Stock 20%,Stock 30%,Stock 40%,Stock 50%,Stock 60%,Stock 70%,Stock 80%,Stock 90%,Stock 100%
Offer premium 0%,4.31,2.57,1.74,1.26,0.94,0.72,0.55,0.42,0.32,0.24,0.17
Offer premium 10%,3.94,2.23,1.46,1.02,0.74,0.54,0.39,0.28,0.19,0.12,0.06
Offer premium 20%,3.57,1.92,1.21,0.81,0.56,0.38,0.25,0.15,0.08,0.01,-0.04
Offer premium 30%,3.21,1.62,0.97,0.62,0.39,0.24,0.13,0.04,-0.02,-0.08,-0.12
Offer premium 40%,2.84,1.35,0.76,0.45,0.25,0.12,0.02,-0.05,-0.11,-0.16,-0.19
Offer premium 50%,2.47,1.09,0.57,0.29,0.12,0.01,-0.07,-0.14,-0.18,-0.22,-0.26
Offer premium 60%,2.10,0.84,0.39,0.15,0.01,-0.09,-0.16,-0.21,-0.25,-0.29,-0.31
Offer premium 70%,1.73,0.61,0.22,0.02,-0.10,-0.18,-0.24,-0.28,-0.31,-0.34,-0.36
Offer premium 80%,1.37,0.39,0.07,-0.10,-0.19,-0.26,-0.31,-0.34,-0.37,-0.39,-0.41
Offer premium 90%,1.00,0.19,-0.07,-0.20,-0.28,-0.33,-0.37,-0.40,-0.42,-0.43,-0.45


In [275]:
acq, tgt = get_companys_datas("AAPL", "SPOT")
sensitivity_accretion_dilution(acq, tgt, verbose=False, steps=2)

,Stock 0%,Stock 20%,Stock 40%,Stock 60%,Stock 80%,Stock 100%
Offer premium 0%,-0.02%,-0.02%,-0.02%,-0.02%,-0.02%,-0.02%
Offer premium 20%,-0.03%,-0.03%,-0.03%,-0.03%,-0.03%,-0.03%
Offer premium 40%,-0.04%,-0.04%,-0.04%,-0.04%,-0.04%,-0.04%
Offer premium 60%,-0.05%,-0.05%,-0.05%,-0.05%,-0.05%,-0.05%
Offer premium 80%,-0.06%,-0.06%,-0.06%,-0.06%,-0.06%,-0.05%
Offer premium 100%,-0.07%,-0.07%,-0.06%,-0.06%,-0.06%,-0.06%


,Stock 0%,Stock 20%,Stock 40%,Stock 60%,Stock 80%,Stock 100%
Offer premium 0%,-0.02,-0.02,-0.02,-0.02,-0.02,-0.02
Offer premium 20%,-0.03,-0.03,-0.03,-0.03,-0.03,-0.03
Offer premium 40%,-0.04,-0.04,-0.04,-0.04,-0.04,-0.04
Offer premium 60%,-0.05,-0.05,-0.05,-0.05,-0.05,-0.05
Offer premium 80%,-0.06,-0.06,-0.06,-0.06,-0.06,-0.05
Offer premium 100%,-0.07,-0.07,-0.06,-0.06,-0.06,-0.06


In [276]:
def accretion_dilution_model(acq, tgt):
    acq_dict, tgt_dict = get_companys_datas(acq, tgt, verbose=True)
    contract_offer(acq_dict, tgt_dict, verbose=True)
    sensitivity_accretion_dilution(acq_dict, tgt_dict)

In [277]:
accretion_dilution_model("AAPl", "NCLH")

values          | AAPL                      | NCLH                     
------------------------------------------------------------
Market cap      | 3,644,938,780,672         | 8,632,590,336            
Price           | 248.96                    | 19.64                    
Shares          | 14681140000               | 455545641                


Parameters:
Offer premium:            | 60.0%
% of cash:                | 50.0%
% of stock:               | 50.0%
tax rate:                 | 40.0%
interest rate:            | 5.0%
% financing fees:         | 2.0%
% transaccion fees:       | 2.0%
% synergies:              | 0%
------------------------------------------------------------

Deal:
Share price:              | 31
Offer Value:              | 14,315,066,223
Money borrowed:           | 7,157,533,111
Financing fees:           | 250,513,659
Shares issued:            | 28,749,731

------------------------------------------------------------

Income:
Accuary net income:       | 124,963,2

,Stock 0%,Stock 10%,Stock 20%,Stock 30%,Stock 40%,Stock 50%,Stock 60%,Stock 70%,Stock 80%,Stock 90%,Stock 100%
Offer premium 0%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Offer premium 10%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Offer premium 20%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Offer premium 30%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Offer premium 40%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Offer premium 50%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Offer premium 60%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Offer premium 70%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Offer premium 80%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Offer premium 90%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%


In [38]:
tgt_ticker = yf.Ticker("NCLH")
for i in tgt_ticker.info:
    print(i)

address1
city
state
zip
country
phone
fax
website
industry
industryKey
industryDisp
sector
sectorKey
sectorDisp
longBusinessSummary
fullTimeEmployees
companyOfficers
auditRisk
boardRisk
compensationRisk
shareHolderRightsRisk
overallRisk
governanceEpochDate
compensationAsOfEpochDate
executiveTeam
maxAge
priceHint
previousClose
open
dayLow
dayHigh
regularMarketPreviousClose
regularMarketOpen
regularMarketDayLow
regularMarketDayHigh
payoutRatio
beta
trailingPE
forwardPE
volume
regularMarketVolume
averageVolume
averageVolume10days
averageDailyVolume10Day
bid
ask
bidSize
askSize
marketCap
nonDilutedMarketCap
fiftyTwoWeekLow
fiftyTwoWeekHigh
allTimeHigh
allTimeLow
priceToSalesTrailing12Months
fiftyDayAverage
twoHundredDayAverage
trailingAnnualDividendRate
trailingAnnualDividendYield
currency
tradeable
enterpriseValue
profitMargins
floatShares
sharesOutstanding
sharesShort
sharesShortPriorMonth
sharesShortPreviousMonthDate
dateShortInterest
sharesPercentSharesOut
heldPercentInsiders
heldPerce

In [72]:
acq, tgt = get_companys_datas("AAPL", "SPOT")
contract_offer(acq, tgt, offer_premium=0.30)

-----AAPL Acquirer---------------
Market cap is 3,644,938,780,672, with 14681140000 shares valued at 249.0$
With and EPS of 8.51182,and this 435617005568

-----SPOT Acquirer---------------
Market cap is 97,704,951,808, with 205832527 shares valued at 482.5$
With and EPS of 13.04623,and this 17186000896

Net accretion_dilution_per_share-0.29725123995688385 wich is -0.03492217175138612 


In [64]:
dict_a = yf.Ticker("AAPl").info
dict_a

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app